# 07 · Escenarios: oferta, demanda, stockout y pricing
Convierte el forecast en decisiones **what-if**. El objetivo no es adivinar un precio óptimo sin evidencia, sino mostrar qué supuestos hacen que el stock se agote antes/después y qué experimento conviene ejecutar.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel
install_feature_mart(); panel = load_monthly_panel()
PROJECT = panel['proyecto'].dropna().iloc[0]
d = panel.loc[panel['proyecto'].eq(PROJECT)].sort_values('periodo_mes').copy()
latest = d.iloc[-1]


## Escenario base
Usamos ritmo neto MA3 como baseline operacional transparente. En producción puede reemplazarse por el champion del notebook 05.

In [ ]:
stock = float(latest['saldo_final_observado'])
base_rate = max(float(latest['mov_neto_ma3']) if pd.notna(latest['mov_neto_ma3']) else 0, 0.01)
scenarios=[]
for demand_shock in [-0.25,-0.10,0,0.10,0.25]:
    for extra_supply in [0,10,25]:
        rate = max(base_rate*(1+demand_shock),0.01)
        effective_stock = stock + extra_supply
        scenarios.append({'shock_demanda':demand_shock,'altas_futuras':extra_supply,'ritmo_neto_mes':rate,'stock_escenario':effective_stock,'meses_a_agotar':effective_stock/rate})
scenario_df = pd.DataFrame(scenarios)
scenario_df.sort_values('meses_a_agotar').head(20)

In [ ]:
pivot = scenario_df.pivot(index='altas_futuras', columns='shock_demanda', values='meses_a_agotar')
fig, ax = plt.subplots(figsize=(9,4))
for altas, row in pivot.iterrows():
    ax.plot([f'{x:+.0%}' for x in pivot.columns], row.values, marker='o', label=f'+{altas} altas')
ax.set_title(f'{PROJECT} · Meses a agotar según shock de demanda y nuevas altas'); ax.set_ylabel('Meses'); ax.set_xlabel('Shock demanda'); ax.legend(); ax.grid(alpha=.2); plt.show()

## Precio: regla de evidencia
Si sólo tenemos `precio_*_actual_ref`, cualquier curva precio-demanda sería contrafactual inventada. El sistema exige una de estas fuentes antes de estimar elasticidad: snapshots de lista por mes, precio efectivo histórico por unidad, o experimento de descuento con grupo de control.

In [ ]:
pricing_gate = pd.DataFrame([
 {'evidencia':'Snapshot mensual precio lista','estado':'REQUERIDO para elasticidad temporal'},
 {'evidencia':'Precio venta histórico','estado':'ÚTIL para willingness-to-pay observado'},
 {'evidencia':'A/B o rollout escalonado de descuento','estado':'PREFERIDO para causalidad'},
 {'evidencia':'Precio actual solamente','estado':'NO suficiente para causalidad'}
])
pricing_gate

## Matriz de acción
- absorción alta + poco stock → evaluar alza/menor descuento;
- absorción baja + mucho stock → investigar precio, canal, producto y macro antes de descontar;
- caídas altas → intervenir conversión/calidad de separación;
- minutas bajas con separaciones sanas → revisar financiamiento, documentación y cierre.

Toda recomendación debe registrar hipótesis, acción, guardrail, resultado y aprendizaje.